# 📗 통계 기초 — 통계적 추론

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

우리는 세상의 모든 것을 다 조사할 수 없습니다. 여론조사는 5천만 명이 아니라 천 명에게 묻고, 공장은 만든 제품을 전부 부숴 보지 않고 몇 개만 시험합니다. **일부(표본)를 보고 전체(모집단)를 추측하는 것** — 이것이 통계적 추론입니다. 이번 시간에는 '표본으로 전체를 어떻게, 얼마나 믿고 말할 수 있는가'의 **직관**을 시뮬레이션으로 몸에 익힙니다.

## ⏪ 복습 — 지난 시간: 기술통계·확률·분포
지난 시간에는 **가진 데이터 자체를 요약**했습니다.
- 대표값·산포도(`mean`·`median`·`std`·`IQR`)로 데이터의 중심과 퍼짐을 봤습니다.
- 확률분포(정규·이항·포아송)와 `68-95-99.7` 법칙, 표준화(Z-score)를 익혔습니다.
- 정규분포에서 `stats.norm.cdf/ppf` 로 확률과 경곗값을 계산했습니다.

기술통계는 **손에 쥔 데이터를 설명**하는 일이었습니다. 이제 한 걸음 더 나아가, **손에 없는 전체를 추측**합니다.

**오늘의 목표**
- [ ] **모집단과 표본**을 구분하고, 표본마다 통계량이 달라지는 **표본분포**를 이해한다.
- [ ] **중심극한정리(CLT)** 를 시뮬레이션으로 확인한다 — n 이 커지면 표본평균이 정규분포로.
- [ ] **표준오차(SE = σ/√n)** 로 표본평균이 얼마나 흔들리는지 잰다.
- [ ] **점추정과 구간추정**의 차이를 알고, 오차를 담은 **신뢰구간**을 두 방법으로 만든다.
- [ ] 신뢰구간을 **올바르게 해석**하고, 그것으로 **어떤 주장(값)이 데이터와 맞는지** 판단한다.
- [ ] **p-value 의 직관**을 잡는다 — 주장이 맞다고 가정했을 때 지금 본 결과가 얼마나 드문가.

In [ ]:
# [제공 코드] 통계적 추론에 쓸 라이브러리와 한글 폰트를 준비합니다.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지
sns.set_theme(font=KOREAN_FONT, rc={'axes.unicode_minus': False})

---
## 왜 추론이 필요한가 — 전수조사는 대개 불가능하다

전체를 남김없이 조사하는 것을 **전수조사**라고 합니다. 하지만 현실에서는
- **너무 많다** — 전국 유권자 전부에게 전화할 수 없습니다.
- **파괴적이다** — 라면의 평균 무게를 알려고 생산된 라면을 전부 뜯어볼 순 없습니다.
- **아직 안 생겼다** — 내년 고객의 평균 구매액은 조사할 대상이 존재하지 않습니다.

그래서 **일부만 뽑아(표본) 전체(모집단)를 추측**합니다. 핵심 용어 두 개를 먼저 나눕니다.

| 구분 | 대상 | 요약값 이름 | 성질 |
|---|---|---|---|
| **모집단(population)** | 알고 싶은 전체 | **모수(parameter)** μ, σ | 고정돼 있지만 보통 **모른다** |
| **표본(sample)** | 실제로 관찰한 일부 | **통계량(statistic)** x̄, s | 표본마다 **달라진다** |

우리의 목표는 **관찰 가능한 통계량(x̄)** 으로 **관찰 불가능한 모수(μ)** 를 추측하는 것입니다.

In [ ]:
# 이번 시간에는 mpg(자동차 연비) 데이터를 '모집단'으로 삼습니다.
# 즉, 이 자동차들이 '세상의 모든 자동차'라고 가정하고, 여기서 표본을 뽑아 볼 것입니다.
population = pd.read_csv('data/mpg.csv')
print('모집단 크기(자동차 수):', len(population))
print('\n[앞부분 5행]'); display(population.head())
print('\n[구조: 열·자료형·결측]'); population.info()
print('\n[수치형 요약]'); display(population.describe())
print('\n[범주형 요약]'); display(population.describe(exclude='number'))

연비(`mpg`)는 오른쪽으로 살짝 꼬리가 긴(왜도 > 0) 분포입니다. 이 **치우친 모집단**에서 표본을 뽑아도 표본평균의 분포가 정규분포로 수렴한다는 것을 곧 눈으로 확인합니다. 지금부터 `population['mpg']` 를 '전체 자동차의 연비'로 보고, **우리는 이 전체를 모른다고 가정**한 채 일부만 뽑아 추측해 나갑니다.

---
# 1. 모집단 vs 표본 — 표본마다 달라지는 '표본분포'

## 왜 필요할까요?
표본을 한 번 뽑아 평균을 내면 그것은 **하나의 숫자**입니다. 그런데 표본을 **다시** 뽑으면 평균이 조금 달라집니다. 또 뽑으면 또 달라집니다. 그렇다면 '표본평균'이라는 값 자체가 하나의 **분포**를 이룬다는 뜻입니다. 이 분포를 **표본분포(sampling distribution)** 라고 합니다.

- **모수 μ**: 전체 자동차의 진짜 평균 연비 — 고정돼 있지만 우리는 모릅니다.
- **통계량 x̄**: 우리가 뽑은 30대의 평균 연비 — 표본을 바꾸면 값이 흔들립니다.
- **표본분포**: '표본평균'이라는 통계량이 이루는 분포. 추론은 이 분포를 이해하는 데서 출발합니다.

> 개념 그림 — 모집단에서 표본을 반복해 뽑으면, 각 표본의 통계량이 모여 하나의 분포를 이룹니다.

<img src="images/표본분포_개념.png" width="780" style="max-width:100%"/>

In [ ]:
mpg_population = population['mpg']
pop_mean = mpg_population.mean()          # 모평균(모수) — 원래는 알 수 없는 '정답'
print('모평균(모수) μ = %.3f' % pop_mean)

# 표본을 여러 번 뽑아 보면 표본평균이 매번 달라진다 (random_state 로 재현 고정)
for state in [1, 2, 3]:
    one_sample = population.sample(n=30, random_state=state)['mpg']
    print('표본 %d (30대): 표본평균 x̄ = %.3f' % (state, one_sample.mean()))

# 표본평균 1000개를 모으면? -> 그것이 이루는 분포가 '표본분포'
rng = np.random.default_rng(42)
pop_values = mpg_population.to_numpy()
sample_means = [rng.choice(pop_values, size=30, replace=True).mean() for _ in range(1000)]

plt.figure(figsize=(8, 4))
sns.histplot(sample_means, bins=30, kde=True)
plt.axvline(pop_mean, color='red', linestyle='--', label='모평균 %.2f' % pop_mean)
plt.title('표본평균 1000개의 분포 — 이것이 표본분포')
plt.xlabel('표본평균(30대씩 뽑은 연비 평균)')
plt.legend()
plt.show()
print('표본평균들의 평균 = %.3f  (모평균 %.3f 에 매우 가깝다)' %
      (np.mean(sample_means), pop_mean))

### 🖐️ 함께 따라하기 — 나만의 표본 뽑아 보기
데모는 **자동차 연비(mpg)** 를 모집단으로 삼았죠. 따라하기는 **다른 도메인 — 음료 제조 공장의 품질검사 기록**(`factory_quality.csv`)으로 연습합니다. 지난 단원에서 쓰던 그 공장 데이터예요.

이번엔 **당도(`당도_brix`)** 를 모집단으로 삼습니다. 당도를 재려면 병을 열어야 하니 **전수조사가 불가능**하죠 — 표본조사가 꼭 필요한 상황입니다.

> ⚠️ 이 셀에서 만드는 `brix_pop`(모집단)과 `brix_mu`(모평균)를 **이후 따라하기에서 계속 씁니다** — 꼭 실행하고 넘어가세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# ※ 데모는 '자동차 연비'를 모집단으로 삼았죠. 이번엔 다른 데이터(공장 당도)로 연습합니다.
# 1) pd.read_csv 로 data/factory_quality.csv 를 읽어 fac 에 담는다
# 2) brix_pop = fac['당도_brix'].dropna() 로 결측을 뺀 당도를 모집단으로 삼는다
# 3) brix_mu = brix_pop.mean() 으로 모평균을 구해 출력한다 (소수 4자리)
#    brix_values = brix_pop.to_numpy() 도 만들어 둔다 (뒤에서 시뮬레이션에 쓴다)
# 4) brix_pop 에서 n=50, random_state=7 로 표본을 뽑아 my_sample 에 담는다
# 5) 표본평균과 (표본평균 - 모평균) 을 출력해 얼마나 빗나갔는지 확인한다

### ✅ 바로 확인 퀴즈
**1.** '모수'와 '통계량'의 차이는 무엇인가요?

<details><summary>정답 보기</summary>

**모수**는 모집단 전체의 요약값(예: 모평균 μ)으로 고정돼 있지만 보통 **알 수 없습니다**. **통계량**은 표본에서 계산한 요약값(예: 표본평균 x̄)으로, 표본을 바꾸면 **값이 달라집니다**. 추론은 통계량으로 모수를 추측하는 일입니다.

</details>

**2.** '표본분포'란 무엇의 분포인가요?

<details><summary>정답 보기</summary>

**통계량(예: 표본평균)** 이 이루는 분포입니다. 같은 크기의 표본을 반복해 뽑아 매번 평균을 내면 그 평균값들이 하나의 분포를 이루는데, 이것이 표본분포입니다. 개별 데이터의 분포와 혼동하지 않는 것이 중요합니다.

</details>

---
# 2. 중심극한정리(CLT) — 표본평균은 정규분포로 모인다

## 왜 필요할까요?
모집단이 정규분포가 아니어도(연비처럼 오른쪽 꼬리가 길어도), **표본크기 n 이 커지면 표본평균의 분포는 정규분포에 가까워집니다.** 이것이 **중심극한정리(Central Limit Theorem)** 입니다. 덕분에 우리는 모집단 모양을 몰라도 표본평균에 대해 정규분포의 언어(신뢰구간 등)를 쓸 수 있습니다.

- 표본평균의 분포는 모평균 μ 를 중심으로 모입니다(치우침이 사라짐).
- n 이 커질수록 분포는 **좁아지고**(흔들림 감소) **종 모양**에 가까워집니다.
- 경험칙: **n ≥ 30 이면 대개 정규 근사가 잘 통합니다.**

<img src="images/중심극한정리.png" width="780" style="max-width:100%"/>

In [ ]:
# 왜곡된 모집단(연비)에서 n=1,5,30,100 으로 각각 표본평균 2000개씩 -> 분포 변화 관찰
rng = np.random.default_rng(42)
pop_values = mpg_population.to_numpy()

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, n in zip(axes.flat, [1, 5, 30, 100]):
    means = [rng.choice(pop_values, size=n, replace=True).mean() for _ in range(2000)]
    sns.histplot(means, bins=30, kde=True, ax=ax)
    ax.axvline(pop_mean, color='red', linestyle='--')
    ax.set_title('표본크기 n=%d — 표본평균 2000개' % n)
    ax.set_xlabel('표본평균')
fig.suptitle('중심극한정리 — n 이 커질수록 표본평균 분포가 정규분포에 가까워진다')
fig.tight_layout()
plt.show()
print('n=1 은 모집단 그대로(오른쪽 꼬리), n=100 은 좁고 대칭인 종 모양')

### 🖐️ 함께 따라하기 — n=50 일 때의 표본평균 분포
공장 당도 모집단(`brix_values`)에서 표본크기 50, 반복 2000회로 표본평균을 모아 히스토그램을 그리고, 모평균 위치에 빨간 점선을 그어 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) rng = np.random.default_rng(1) 로 난수 생성기를 만든다
# 2) rng.choice(brix_values, size=50, replace=True).mean() 을 2000번 반복해 리스트에 담는다
# 3) plt.figure() 로 새 도화지를 연 뒤 sns.histplot(..., kde=True) 로 그린다
# 4) plt.axvline 으로 모평균(brix_mu)에 빨간 점선을 긋고 제목·범례를 단다

### ✅ 바로 확인 퀴즈
**1.** 모집단이 정규분포가 아니면 중심극한정리는 쓸 수 없다? (O/X)

<details><summary>정답 보기</summary>

**X.** 중심극한정리의 핵심은 **모집단 모양과 무관하게** 표본크기 n 이 커지면 **표본평균**의 분포가 정규분포로 수렴한다는 것입니다. 연비처럼 치우친 모집단에서도 성립합니다.

</details>

**2.** CLT 가 말하는 정규 근사는 '개별 데이터'가 정규분포가 된다는 뜻인가요?

<details><summary>정답 보기</summary>

아닙니다. 정규분포에 가까워지는 것은 **표본평균(통계량)의 분포**이지 개별 데이터의 분포가 아닙니다. 개별 연비 값들은 여전히 오른쪽 꼬리가 긴 모양 그대로입니다.

</details>

---
# 3. 표준오차(SE = σ/√n) — 표본평균은 얼마나 흔들리나

## 왜 필요할까요?
표본평균이 표본마다 달라진다면, **얼마나** 달라지는지를 숫자로 재고 싶습니다. 표본평균의 표준편차를 **표준오차(Standard Error)** 라고 하며, 다음 공식으로 계산합니다.

<div style="text-align:right">$SE = \dfrac{\sigma}{\sqrt{n}}$</div>

- 분모에 √n 이 있으므로 **표본이 클수록 표준오차는 작아집니다**(추정이 더 안정적).
- 실무에서는 모표준편차 σ 를 모르므로 표본표준편차 s 로 대신해 `SE ≈ s/√n` 을 씁니다.
- `stats.sem(표본)` 이 이 값을 자동으로 계산해 줍니다(s/√n 과 같음).

핵심 감각: **표본크기를 4배로 늘리면 표준오차는 √4 = 2, 즉 절반으로 줄어듭니다.** 정밀도를 2배 높이려면 표본을 4배 모아야 한다는 뜻입니다.

In [ ]:
sample30 = population.sample(n=30, random_state=42)['mpg']
n = len(sample30)
s = sample30.std(ddof=1)                  # 표본표준편차
se_manual = s / np.sqrt(n)                # 직접 계산한 표준오차
se_scipy = stats.sem(sample30)            # scipy 로 계산한 표준오차
print('표본표준편차 s = %.4f' % s)
print('직접 계산 SE = s/√n = %.4f' % se_manual)
print('stats.sem()  SE = %.4f  (둘이 같다)' % se_scipy)

# n 이 커지면 SE 는 어떻게 줄어드나 — 이론값 σ/√n 으로 √n 관계 확인
pop_std = mpg_population.std(ddof=0)      # 모표준편차 σ (지금은 모집단을 안다고 가정)
print('\n모표준편차 σ = %.3f 로 본 이론 표준오차:' % pop_std)
for size in [30, 120, 480]:
    print('  n=%3d → SE = σ/√n = %.4f' % (size, pop_std / np.sqrt(size)))
print('n 이 4배(30→120)가 되면 SE 는 절반으로 줄어든다')

### 🖐️ 함께 따라하기 — 표본을 키우면 표준오차가 주는지 확인
공장 당도 모집단에서 `random_state=42` 로 n=30 과 n=60 표본을 각각 뽑아 `stats.sem` 으로 표준오차를 구해 비교해 봅니다(n 이 큰 쪽이 더 작아야 합니다).

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) brix_pop 에서 n=30, random_state=42 로 표본을 뽑아 brix30 에 담는다
# 2) 같은 방식으로 n=60 표본을 brix60 에 담는다
# 3) stats.sem 으로 두 표준오차를 각각 구해 나란히 출력한다
# 4) (n=60 SE) / (n=30 SE) 비율도 출력해, 이론값 1/√2 ≈ 0.707 과 비교한다

### ✅ 바로 확인 퀴즈
**1.** 표준오차 공식 `SE = σ/√n` 에서 표본크기 n 이 커지면 SE 는 어떻게 되나요?

<details><summary>정답 보기</summary>

**작아집니다.** n 이 분모의 √ 안에 있으므로, 표본이 클수록 표본평균의 흔들림(표준오차)이 줄어 추정이 더 정밀해집니다. n 을 4배로 하면 SE 는 절반이 됩니다.

</details>

**2.** `stats.sem(sample)` 은 무엇을 계산하나요?

<details><summary>정답 보기</summary>

표본의 **표준오차**, 즉 `s/√n`(표본표준편차를 √n 으로 나눈 값)을 계산합니다. 직접 `sample.std(ddof=1)/np.sqrt(len(sample))` 로 구한 값과 같습니다.

</details>

---
# 4. 점추정 vs 구간추정 — 하나의 숫자에서 범위로

## 왜 필요할까요?
표본평균 하나로 모평균을 '이 값이다'라고 찍는 것을 **점추정(point estimation)** 이라고 합니다. 간단하지만 문제가 있습니다 — 표본을 바꾸면 그 값이 흔들리고, **얼마나 믿을 만한지**를 전혀 담지 못합니다.

| 방식 | 결과 | 한계 / 장점 |
|---|---|---|
| **점추정** | x̄ = 22.6 처럼 **숫자 하나** | 간단하지만 불확실성을 못 담고, 딱 맞기 어렵다 |
| **구간추정** | [19.5, 25.6] 처럼 **범위** | 오차를 함께 담아 '이 범위 안일 것'이라 말할 수 있다 |

그래서 실무에서는 점추정값에 **오차범위**를 붙여 구간으로 말합니다. 다음 섹션의 **신뢰구간**이 바로 그 구간추정입니다.

In [ ]:
sample30 = population.sample(n=30, random_state=42)['mpg']
point_estimate = sample30.mean()
print('점추정: 표본평균 %.3f 로 모평균을 추정' % point_estimate)
print('실제 모평균 %.3f — 딱 맞지 않는다 (오차 %.3f)' %
      (pop_mean, point_estimate - pop_mean))

# 표본을 바꿀 때마다 점추정값이 흔들린다 -> '숫자 하나'로는 불확실성을 못 담는다
print('\n표본을 바꾸면 점추정값이 흔들린다:')
for state in [10, 20, 30, 40]:
    est = population.sample(n=30, random_state=state)['mpg'].mean()
    print('  표본평균 = %.3f' % est)

### 🖐️ 함께 따라하기 — 점추정의 오차 재 보기
공장 당도 모집단에서 `random_state=99` 로 n=30 표본을 뽑아 점추정값(표본평균)을 구하고, 모평균과의 오차의 절댓값을 출력해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) brix_pop 에서 n=30, random_state=99 로 표본을 뽑아 brix_99 에 담는다
# 2) 그 평균(점추정값)을 point_99 에 담아 출력한다
# 3) abs(point_99 - brix_mu) 로 모평균과의 오차 절댓값을 출력한다

### ✅ 바로 확인 퀴즈
**1.** 점추정의 가장 큰 한계는 무엇인가요?

<details><summary>정답 보기</summary>

**불확실성(오차)을 담지 못합니다.** 숫자 하나만 제시하므로 그 추정이 얼마나 믿을 만한지, 실제 모수와 얼마나 떨어져 있을 수 있는지 알려 주지 못합니다. 게다가 표본이 바뀌면 값이 흔들립니다.

</details>

**2.** 구간추정이 점추정보다 나은 점은 무엇인가요?

<details><summary>정답 보기</summary>

점추정값에 **오차범위를 더해 범위로** 말하므로, '모수가 대략 이 범위 안에 있을 것'이라는 **불확실성의 크기**까지 함께 전달할 수 있습니다.

</details>

---
# 5. 신뢰구간 — 오차를 담은 구간 만들기 (핵심)

## 왜 필요할까요?
**신뢰구간(Confidence Interval)** 은 점추정값 좌우로 오차범위를 붙인 구간입니다. '95% 신뢰구간'이라면, **같은 방식으로 구간을 아주 많이 만들면 그중 약 95%가 진짜 모수를 포함**한다는 뜻입니다.

이 단원에서는 신뢰구간을 **두 가지 길**로 만듭니다(정식 검정함수는 다음 단원).
1. **부트스트랩 시뮬레이션** — 표본에서 복원추출을 반복해 표본평균 분포를 만들고, 그 가운데 95% 구간(2.5% ~ 97.5% 지점)을 잘라낸다. 공식을 몰라도 되는 직관적 방법.
2. **공식** — 표본평균 ± (임계값 × 표준오차). `stats.norm.interval`(대표본) 또는 `stats.t.interval`(소표본)로 계산.

<img src="images/신뢰구간_개념.png" width="780" style="max-width:100%"/>

In [ ]:
sample30 = population.sample(n=30, random_state=42)['mpg']
sample_values = sample30.to_numpy()

# 방법① 부트스트랩: 표본에서 복원추출을 반복해 표본평균 분포를 만든다
rng = np.random.default_rng(42)
boot_means = [rng.choice(sample_values, size=len(sample_values), replace=True).mean()
              for _ in range(2000)]
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
print('부트스트랩 95%% 신뢰구간 = [%.3f, %.3f]' % (ci_low, ci_high))

plt.figure(figsize=(8, 4))
sns.histplot(boot_means, bins=30, kde=True)
plt.axvline(ci_low, color='red', linestyle='--')
plt.axvline(ci_high, color='red', linestyle='--', label='95% 신뢰구간')
plt.axvline(pop_mean, color='green', linestyle='-', label='모평균 %.2f' % pop_mean)
plt.title('부트스트랩으로 만든 표본평균 분포와 95% 신뢰구간')
plt.xlabel('표본평균(복원추출)')
plt.legend()
plt.show()

## 방법② 공식 — 표본평균 ± (임계값 × 표준오차)

표본평균의 분포가 정규분포에 가깝다는 것(CLT)을 이용하면, 공식으로 바로 구간을 낼 수 있습니다.

<div style="text-align:right">$\bar{x} \pm (\text{임계값}) \times SE$</div>

- **대표본**(n 이 충분히 큼): 정규분포 임계값 사용 → `stats.norm.interval(0.95, loc=x̄, scale=SE)`
- **소표본**(n 이 작고 σ 를 모름): **t분포** 임계값 사용 → `stats.t.interval(0.95, df=n-1, loc=x̄, scale=SE)`

**t분포**는 정규분포와 비슷하지만 **꼬리가 더 두껍습니다.** 표본이 작아 표준편차 추정이 불안정한 만큼, 구간을 조금 더 넓게 잡아 안전하게 만듭니다. 자유도(df = n − 1)가 커질수록 t분포는 정규분포에 가까워집니다.

<img src="images/t분포_비교.png" width="780" style="max-width:100%"/>

In [ ]:
mean_hat = sample30.mean()
se = stats.sem(sample30)
n = len(sample30)

# 정규분포 기반(대표본) 과 t분포 기반(소표본)
ci_norm = stats.norm.interval(0.95, loc=mean_hat, scale=se)
ci_t = stats.t.interval(0.95, df=n - 1, loc=mean_hat, scale=se)
print('정규분포 기반 95%% CI = [%.3f, %.3f]' % ci_norm)
print('t분포 기반   95%% CI = [%.3f, %.3f]' % ci_t)
print('부트스트랩   95%% CI = [%.3f, %.3f]' % (ci_low, ci_high))
print('\n세 방법의 결과가 비슷하다 — 같은 불확실성을 서로 다른 길로 계산한 것')
print('t분포 구간이 가장 넓다 — 소표본의 불확실성을 반영해 조금 더 안전하게 잡는다')

## 신뢰구간의 올바른 해석 — 흔한 오해 바로잡기

'95% 신뢰구간이 [19.5, 25.6]' 이라는 말의 뜻을 정확히 새깁니다.

- ❌ **틀린 해석**: "모평균이 이 구간에 있을 확률이 95%다." — 모평균은 고정된 값이라 '확률'로 말할 수 없습니다.
- ✅ **옳은 해석**: "**같은 방식으로 구간을 100번 만들면 그중 약 95번이 모평균을 포함**한다." 지금 만든 이 구간 하나는 그 95번 중 하나이길 기대하는 것입니다.

아래 시뮬레이션은 신선한 표본으로 95% 신뢰구간을 100번 만들어, 실제로 몇 개가 모평균을 담는지 셉니다.

<img src="images/신뢰구간_유형.png" width="780" style="max-width:100%"/>

In [ ]:
# 같은 방식으로 100번 만들면 약 95번이 모평균을 포함하는지 직접 세어 본다
rng = np.random.default_rng(42)
pop_values = mpg_population.to_numpy()
cover_count = 0
intervals = []
for _ in range(100):
    smp = rng.choice(pop_values, size=30, replace=True)
    low, high = stats.norm.interval(0.95, loc=smp.mean(), scale=stats.sem(smp))
    intervals.append((low, high))
    if low <= pop_mean <= high:
        cover_count += 1
print('100번 만든 95%% 신뢰구간 중 모평균을 포함한 것: %d 개' % cover_count)

# 앞 30개 구간을 그려 모평균 포함 여부를 색으로 표시(회색=포함, 빨강=놓침)
plt.figure(figsize=(8, 6))
for i, (low, high) in enumerate(intervals[:30]):
    hit = low <= pop_mean <= high
    plt.plot([low, high], [i, i], color=('gray' if hit else 'red'), linewidth=2)
plt.axvline(pop_mean, color='green', linestyle='--', label='모평균 %.2f' % pop_mean)
plt.title('30개의 95% 신뢰구간 — 빨간 구간만 모평균을 놓쳤다')
plt.xlabel('연비'); plt.ylabel('구간 번호')
plt.legend()
plt.show()

## 신뢰수준과 구간 너비 — 맞바꿈(trade-off)

더 확실하게(신뢰수준을 높게) 말하려면 구간을 **더 넓게** 잡아야 합니다. 90% → 95% → 99% 로 갈수록 구간이 넓어집니다. 좁으면서 확실한 구간은 공짜로 얻어지지 않습니다.

- **좁은 구간**: 정보가 뾰족하지만 틀릴 위험이 큽니다.
- **넓은 구간**: 안전하지만 '어디쯤'이라는 정보가 흐려집니다.

<img src="images/신뢰수준_비교.png" width="780" style="max-width:100%"/>

In [ ]:
mean_hat = sample30.mean()
se = stats.sem(sample30)
n = len(sample30)
print('신뢰수준이 높아질수록 구간이 넓어진다:')
for level in [0.90, 0.95, 0.99]:
    low, high = stats.t.interval(level, df=n - 1, loc=mean_hat, scale=se)
    print('  %2.0f%% 신뢰구간 = [%.3f, %.3f]  (폭 %.3f)' %
          (level * 100, low, high, high - low))

### 🖐️ 함께 따라하기 — 90% 신뢰구간을 두 방법으로
공장 당도 표본(n=30)으로 **90%** 신뢰구간을 ① `stats.t.interval` 공식과 ② 부트스트랩 `np.percentile([...], [5, 95])` 두 방법으로 각각 구해 비교해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 0) brix30 = brix_pop.sample(n=30, random_state=42) 로 표본을 만든다
# ① 공식 방법: stats.t.interval 로 90% 신뢰구간을 구한다
#    - 신뢰수준 0.90, df=len(brix30)-1, loc=표본평균, scale=stats.sem(brix30)
# ② 부트스트랩 방법:
#    - rng = np.random.default_rng(42) 로 생성기를 만든다
#    - brix30 에서 복원추출한 표본평균을 2000번 모은다
#    - np.percentile(boot, [5, 95]) 로 90% 구간의 양 끝을 구한다
# 두 구간을 각각 출력해 비슷한지 비교한다

### ✅ 바로 확인 퀴즈
**1.** "95% 신뢰구간 [19.5, 25.6]" 을 올바르게 해석하면?

<details><summary>정답 보기</summary>

"**같은 방식으로 구간을 여러 번 만들면 그중 약 95%가 모평균을 포함**한다"는 뜻입니다. '모평균이 이 구간에 있을 확률이 95%'라는 말은 틀렸습니다 — 모평균은 고정된 값이라 확률로 말하지 않습니다.

</details>

**2.** 신뢰수준을 95%에서 99%로 높이면 구간의 너비는 어떻게 되나요?

<details><summary>정답 보기</summary>

**넓어집니다.** 더 확실하게(더 자주 모수를 포함하도록) 말하려면 구간을 더 넓게 잡아야 하기 때문입니다. 확실함과 좁음은 맞바꿈 관계입니다.

</details>

**3.** 표본이 작을 때 정규분포 대신 t분포를 쓰는 이유는?

<details><summary>정답 보기</summary>

표본이 작으면 표준편차 추정이 불안정합니다. t분포는 정규분포보다 **꼬리가 두꺼워** 구간을 조금 더 넓게(안전하게) 잡아 이 불확실성을 반영합니다. 자유도(n−1)가 커지면 t분포는 정규분포에 가까워집니다.

</details>

---
# 6. 신뢰구간으로 주장 판단하기

## 왜 필요할까요?
누군가 "전체 자동차의 평균 연비는 **27**이다"라고 주장했다고 합시다. 우리는 전체를 모르지만, 앞에서 배운 **신뢰구간**으로 이 주장이 데이터와 맞는지 판단할 수 있습니다.

- 표본으로 **95% 신뢰구간**을 구합니다 (모평균이 있을 만한 범위).
- 주장한 값이 그 **구간 안에 있으면** → 그 주장은 데이터와 **모순되지 않습니다**(그 값이 모평균일 수 있음).
- 주장한 값이 **구간 밖에 있으면** → 그 주장은 데이터와 **잘 맞지 않습니다**(그 값이 모평균일 가능성이 낮음).

> 신뢰구간은 "모평균이 있을 만한 범위"입니다. 그 범위 **밖**의 값을 누가 주장하면, 우리 데이터는 그 주장을 **뒷받침하지 않는** 셈입니다. 표본이 크고 구간이 좁을수록 판단이 더 또렷해집니다.

In [ ]:
# 주장: '전체 자동차의 평균 연비 = 27'.  이 주장이 데이터와 맞는지 신뢰구간으로 판단한다.
sample30 = population.sample(n=30, random_state=42)['mpg']
observed_mean = sample30.mean()
sem30 = stats.sem(sample30)
ci_low, ci_high = stats.t.interval(0.95, df=len(sample30) - 1, loc=observed_mean, scale=sem30)
print(f'표본평균 = {observed_mean:.2f}')
print(f'95% 신뢰구간 = [{ci_low:.2f}, {ci_high:.2f}]')

claim = 27.0
inside = ci_low <= claim <= ci_high
print(f'주장 {claim:.0f} 이 95% 신뢰구간 안에 있나? {inside}')
print('→ 주장 27 은 구간 밖 → 이 주장은 데이터와 잘 맞지 않는다 (그 값이 모평균일 가능성이 낮다)')

plt.figure(figsize=(9, 3))
plt.axvspan(ci_low, ci_high, alpha=0.2, color='skyblue', label='95% 신뢰구간')
plt.axvline(observed_mean, color='blue', label=f'표본평균 {observed_mean:.2f}')
plt.axvline(claim, color='red', linestyle='--', label=f'주장 {claim:.0f}')
plt.title('신뢰구간으로 주장 판단 — 주장값이 구간 밖이면 데이터와 안 맞음')
plt.xlabel('연비 평균')
plt.yticks([])
plt.legend()
plt.show()

### 🖐️ 함께 따라하기 — 공장의 주장 판단하기
공장 품질팀이 **"우리 라인의 평균 당도는 규격 중심값 11.0 brix 다"** 라고 주장합니다. 당도 표본(n=30)으로 95% 신뢰구간을 만들어 이 주장이 데이터와 맞는지 판단해 봅시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) brix30 = brix_pop.sample(n=30, random_state=42) 로 표본을 만든다
# 2) stats.t.interval(0.95, df=len(brix30)-1, loc=표본평균, scale=stats.sem(brix30)) 로
#    95% 신뢰구간을 구해 b_low, b_high 에 담고 출력한다
# 3) claim_brix = 11.0 으로 주장 값을 정한다
# 4) b_low <= claim_brix <= b_high 로 구간 안에 있는지 확인해 출력한다
# 5) 구간 안이면 '데이터와 모순되지 않는다' 로 해석한다

### ✅ 바로 확인 퀴즈
**1.** 어떤 주장값이 95% 신뢰구간 **밖**에 있으면 어떻게 해석하나요?

<details><summary>정답 보기</summary>

그 주장은 데이터와 **잘 맞지 않습니다** — 표본이 말하는 "모평균이 있을 만한 범위" 밖의 값이라, 그 값이 진짜 모평균일 가능성이 낮습니다. 반대로 구간 **안**이면 데이터와 모순되지 않습니다.

</details>

**2.** 신뢰수준을 99%로 높이면 구간은 넓어집니다. 그러면 주장을 '구간 밖'으로 판정하기가 더 쉬워질까요, 어려워질까요?

<details><summary>정답 보기</summary>

**더 어려워집니다.** 구간이 넓어질수록 더 많은 값이 구간 안에 들어와 '모순되지 않음'으로 판정되기 쉽습니다. 확신을 높이면(신뢰수준↑) 그만큼 판단이 보수적이 됩니다.

</details>

> **다음 단원으로의 다리**: 여기서는 신뢰구간이 주장값을 포함하는지로 주장을 판단했습니다. 다음 단원(**가설검정**)에서는 이 판단을 **정식 검정 절차**로 자동화합니다 — 주장을 세우고 "그 주장이 맞다면 이런 관측이 나올 가능성"을 수치로 계산해 판정합니다. 신뢰구간으로 판단하는 것과 정식 검정은 사실 **동전의 양면**입니다.

---
# 7. p-value — 어긋난 '정도'를 숫자로

## 왜 필요할까요?
바로 앞에서 신뢰구간으로 주장을 판단했습니다. 그런데 신뢰구간은 **"안이냐 밖이냐"** 만 답합니다.

우리 구간이 [19.54, 25.60] 이라면 주장 **26** 도 밖이고 주장 **27** 도 밖입니다. 그럼 두 주장이 **똑같이** 안 맞는 걸까요? 그럴 리 없죠 — 27 이 더 멀리 어긋나 있습니다. 이 **어긋난 정도**를 숫자 하나로 재는 것이 **p-value** 입니다.

## p-value 란
> **주장이 옳다고 가정했을 때, 지금 우리가 본 것만큼(또는 그보다 더) 극단적인 결과가 우연히 나올 확률**

- **p 가 작다** → 주장이 맞는 세상에서는 좀처럼 안 나올 일이 일어났다 → **주장을 의심**할 근거.
- **p 가 크다** → 주장이 맞아도 흔히 나올 수 있는 결과다 → 주장과 **모순되지 않는다**.

<img src="images/p_value_개념.png" width="820" style="max-width:100%"/>

## 방법① 시뮬레이션 — '주장이 맞는 세상'을 만들어 본다

공식을 외우기 전에, p-value 가 무엇인지 **직접 만들어** 봅시다. 순서는 이렇습니다.

1. 주장(모평균 = 27)이 **정말 맞는 가상의 세상**을 만든다 — 모집단을 평행이동해 평균이 27이 되게.
2. 그 세상에서 n=30 표본을 **10000번** 뽑아 표본평균 분포를 만든다.
3. 우리가 실제로 관측한 표본평균이 그 분포의 어디쯤인지 보고, <strong>그만큼 극단적인 경우가 몇 %</strong>인지 센다. 그 비율이 곧 p-value 다.

In [ ]:
sample30 = population.sample(n=30, random_state=42)['mpg']
observed = sample30.mean()
claim = 27.0

# ① 주장(모평균=27)이 맞는 '가상의 세상' — 모집단을 통째로 평행이동
shifted = mpg_population.to_numpy() - pop_mean + claim

# ② 그 세상에서 n=30 표본을 10000번 뽑아 표본평균을 모은다
rng = np.random.default_rng(42)
sim_means = np.array([rng.choice(shifted, size=30, replace=True).mean()
                      for _ in range(10000)])

# ③ 관측값만큼(또는 그보다 더) 주장에서 멀리 떨어진 경우의 비율
gap = abs(observed - claim)
p_sim = (np.abs(sim_means - claim) >= gap).mean()
print('관측된 표본평균 = %.3f  (주장 %.0f 에서 %.3f 만큼 떨어짐)' % (observed, claim, gap))
print('시뮬레이션 p-value = %.4f' % p_sim)

plt.figure(figsize=(9, 4))
sns.histplot(sim_means, bins=40, color='steelblue')
plt.axvline(claim, color='green', linewidth=2, label='주장 %.0f' % claim)
plt.axvline(observed, color='red', linestyle='--', linewidth=2,
            label='관측 표본평균 %.2f' % observed)
plt.title('주장이 맞다고 가정한 세상에서의 표본평균 10000개')
plt.xlabel('표본평균')
plt.legend()
plt.show()
print('빨간 선이 분포의 왼쪽 끝 바깥에 있습니다.')
print('주장이 맞다면 우리가 본 결과는 좀처럼 나오지 않을 결과라는 뜻입니다.')

## 방법② 공식 — t 통계량으로 바로 계산

시뮬레이션 없이도 **표본평균이 주장값에서 표준오차 몇 칸 떨어졌는지**를 재면 p 를 바로 구할 수 있습니다.

<div style="text-align:right">$t = \dfrac{\bar{x} - \mu_0}{SE}$</div>

- `t` 가 0에서 멀수록(양쪽 어디로든) 주장과 어긋난 것 → p 는 작아집니다.
- 양쪽 방향을 다 따지므로 **2를 곱합니다**(양측): `p = 2 * stats.t.sf(abs(t), df=n-1)`.
- `sf` 는 지난 시간에 배운 '초과 확률'(1 − cdf) 입니다. 꼬리 넓이를 그대로 구해 줍니다.

In [ ]:
xbar = sample30.mean()
se = stats.sem(sample30)
n = len(sample30)
ci_low, ci_high = stats.t.interval(0.95, df=n - 1, loc=xbar, scale=se)
print('표본평균 = %.3f / 표준오차 = %.4f' % (xbar, se))
print('95%% 신뢰구간 = [%.3f, %.3f]' % (ci_low, ci_high))

print('\n%-8s %-10s %-10s %s' % ('주장값', '신뢰구간', 't 통계량', 'p-value'))
for cl in [23.0, 26.0, 27.0]:
    t_stat = (xbar - cl) / se
    p_val = 2 * stats.t.sf(abs(t_stat), df=n - 1)
    where = '안' if ci_low <= cl <= ci_high else '밖'
    print('%-8.1f %-10s %-10.3f %.4f' % (cl, where, t_stat, p_val))

print('\n23 은 구간 안이고 p 도 크다 — 데이터와 모순되지 않는다.')
print('26 과 27 은 둘 다 구간 밖이지만, 27 의 p 가 훨씬 작다 — 더 심하게 어긋난다.')
print('신뢰구간은 안/밖만 알려 주지만, p-value 는 그 정도까지 알려 준다.')

print('\n[시뮬레이션과 비교] 앞의 재표집으로 구한 p 는 %.4f, 공식으로 구한 p 는 %.4f 였다.' %
      (p_sim, 2 * stats.t.sf(abs((xbar - 27.0) / se), df=n - 1)))
print('값이 딱 같지는 않다 — 재표집은 모집단의 실제 모양을 그대로 쓰고,')
print('공식은 t분포 근사를 쓰기 때문이다. 중요한 것은 둘 다 0.05 보다 훨씬 작아')
print('같은 결론(주장 27 은 데이터와 잘 맞지 않는다)에 이른다는 점이다.')

## 신뢰구간과 p-value — 동전의 양면

둘은 다른 이야기가 아닙니다. **95% 신뢰구간 밖에 있다 ⟺ p < 0.05** 입니다. 같은 판단을 '범위'로 말하느냐 '정도'로 말하느냐의 차이일 뿐이에요.

<img src="images/p_value와_신뢰구간.png" width="820" style="max-width:100%"/>

실제로 신뢰구간의 **경계값**에서 p 를 재 보면 정확히 0.05 가 나옵니다.

In [ ]:
for edge, name in [(ci_low, '하한'), (ci_high, '상한')]:
    p_edge = 2 * stats.t.sf(abs((xbar - edge) / se), df=n - 1)
    print('95%% 신뢰구간 %s %.3f 에서의 p = %.4f' % (name, edge, p_edge))
print('\n경계에서 정확히 0.05 — 구간 밖이면 p < 0.05, 구간 안이면 p > 0.05 다.')

## p-value 를 오해하지 않기 (아주 중요)

p-value 는 통계에서 **가장 많이 오해받는 숫자**입니다. 네 가지만 확실히 해 둡시다.

| 흔한 오해 | 바로잡기 |
|---|---|
| ❌ p 는 '주장이 맞을 확률'이다 | p 는 **주장이 맞다고 가정한 뒤** 계산한 값입니다. 주장 자체의 확률이 아닙니다. |
| ❌ p 가 크면 주장이 맞다는 증명이다 | **'틀렸다는 증거가 없다'** 일 뿐입니다. 증거 없음 ≠ 없다는 증거. |
| ❌ p < 0.05 면 중요한 발견이다 | 0.05 는 **관례**일 뿐 자연법칙이 아닙니다. 0.049 와 0.051 은 사실상 같습니다. |
| ❌ p 가 작으면 차이가 크다 | p 는 **차이의 크기를 말하지 않습니다**. n 이 아주 크면 사소한 차이도 p 가 작아집니다. |

> 그래서 실무에서는 p-value 하나만 보고 결론짓지 않고, **효과의 크기**(얼마나 차이 나는가)와 **신뢰구간**(어느 범위인가)을 늘 함께 봅니다.

### 🖐️ 함께 따라하기 — 공장의 두 주장을 p-value 로 비교
공장에서 두 사람이 서로 다른 주장을 합니다.

- 생산팀: "평균 당도는 **11.2** brix 다"
- 품질팀: "아니다, **10.9** brix 다"

당도 표본(n=30)으로 **두 주장의 p-value 를 각각 구해** 어느 쪽이 데이터와 더 안 맞는지 비교해 봅시다. (둘 다 95% 신뢰구간 밖이지만, 어긋난 정도는 다릅니다.)

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) brix30 = brix_pop.sample(n=30, random_state=42) 로 표본을 만든다
# 2) 표본평균(xb)·표준오차(se_b)·표본크기(n_b)를 구한다
# 3) stats.t.interval 로 95% 신뢰구간(lo_b, hi_b)을 구해 출력한다
# 4) 주장값 11.2 와 10.9 각각에 대해 반복문으로
#    - t = (xb - 주장값) / se_b
#    - p = 2 * stats.t.sf(abs(t), df=n_b - 1)
#    - 신뢰구간 안/밖 여부와 p 를 함께 출력한다
# 5) 어느 주장이 데이터와 더 안 맞는지 p 로 비교해 말해 본다

### ✅ 바로 확인 퀴즈
**1.** "p = 0.03" 을 올바르게 읽으면?

<details><summary>정답 보기</summary>

"**주장이 맞다고 가정하면**, 지금 본 것만큼 극단적인 결과가 나올 확률이 3%" 라는 뜻입니다. '주장이 맞을 확률이 3%'가 **아닙니다** — p 는 주장을 참으로 가정한 뒤 계산한 값이니까요.

</details>

**2.** 어떤 주장값이 95% 신뢰구간 **안**에 있다면, 그 주장의 p-value 는 0.05 보다 클까요 작을까요?

<details><summary>정답 보기</summary>

**0.05 보다 큽니다.** 신뢰구간 안 ⟺ p > 0.05, 신뢰구간 밖 ⟺ p < 0.05 로 둘은 정확히 맞물립니다. 구간의 경계값에서 p 는 정확히 0.05 입니다.

</details>

**3.** p 가 0.6 으로 크게 나왔습니다. "주장이 옳다는 것이 증명됐다"고 말해도 될까요?

<details><summary>정답 보기</summary>

**안 됩니다.** p 가 크다는 것은 "주장이 틀렸다고 볼 **근거가 없다**"는 뜻이지 "주장이 옳다"는 증명이 아닙니다. 표본이 작아 판단력이 약해서 p 가 클 수도 있거든요. **증거 없음과 없다는 증거는 다릅니다.**

</details>

---
## 🚀 응용 클론코딩 — 표본 하나로 추론 리포트 쓰기

오늘 배운 것을 **한 흐름**으로 이어 봅시다: 표본추출 → 점추정 → 표준오차 → 신뢰구간 → 주장 판단(p-value).

**미션**: 공장에서 당도를 **40배치만** 검사했다고 하고, 그 표본 하나로 "모평균은 얼마쯤인가"와 "규격 중심 11.0 이라는 주장은 타당한가"에 답하는 **짧은 추론 리포트**를 만듭니다.

> 실무에서 통계 리포트는 대개 이 다섯 줄로 끝납니다. 오늘의 결론이자 다음 시간의 출발점입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) brix_pop 에서 n=40, random_state=2025 로 표본을 뽑아 smp 에 담는다
# 2) 점추정값(표본평균)을 구해 출력한다
# 3) stats.sem 으로 표준오차를 구해 출력한다
# 4) stats.t.interval 로 95% 신뢰구간과 그 폭을 구해 출력한다
# 5) 주장 11.0 에 대해 신뢰구간 안/밖과 p-value 를 구해 출력한다
# 6) 위 숫자들을 근거로 '추론 리포트' 를 서너 문장으로 print 한다
# 7) 마지막으로 진짜 모평균(brix_mu)과 비교해, 신뢰구간이 실제로 모평균을 포함했는지 확인한다

---
## 이번 강의 정리

| 개념 | 핵심 | 도구 |
|---|---|---|
| 모집단 vs 표본 | 모수 μ(고정·미지) 를 통계량 x̄(가변) 로 추측 | `df.sample(n, random_state)` |
| 표본분포 | 통계량(표본평균)이 이루는 분포 | 리샘플링 반복 + `histplot` |
| 중심극한정리 | n 이 커지면 표본평균 분포 → 정규분포(모양 무관) | `rng.choice` 시뮬레이션 |
| 표준오차 | 표본평균의 흔들림 `SE = σ/√n`; n 4배 → SE 절반 | `stats.sem` |
| 점추정 vs 구간추정 | 숫자 하나 → 오차를 담은 범위 | 표본평균, 신뢰구간 |
| 신뢰구간 | 100번 만들면 약 95번이 모수 포함 | 부트스트랩 `np.percentile`, `stats.norm/t.interval` |
| 신뢰수준-너비 | 더 확실할수록 구간은 넓어진다(맞바꿈) | `interval(level, ...)` |
| 주장 판단 | 신뢰구간이 주장값을 **포함하지 않으면** 그 주장은 데이터와 잘 안 맞음 | `stats.norm/t.interval` |
| p-value | 주장이 맞다고 가정했을 때 지금 본 결과가 나올 확률. 작을수록 주장을 의심 | `stats.t.sf` (양측 ×2) |
| 구간 ⟺ p | 95% 구간 밖 ⟺ p < 0.05. 같은 판단의 두 표현 | 경계에서 p = 0.05 |

이제 여러분은 **표본으로 전체를 추측**하고(추정), 그 추측에 **얼마나 확신하는지**를 신뢰구간으로 말하며, 어떤 주장이 데이터와 맞는지를 **안/밖(신뢰구간)** 과 **정도(p-value)** 두 가지로 판단할 수 있습니다.

## ⏭️ 예고 — 다음 시간: 가설검정·회귀 (추정을 정식 검정으로)
오늘은 표본분포·중심극한정리·신뢰구간으로 **표본에서 전체를 추정**하고, 신뢰구간으로 **주장을 판단**해 봤습니다. 마지막에는 어긋난 **정도**를 재는 **p-value** 의 직관까지 잡았습니다. 다음 시간에는 이 판단을 **정식 가설검정**으로 자동화합니다 — 귀무가설·대립가설을 세우고 `ttest_ind`·`f_oneway`·`chi2_contingency` 같은 검정 함수로 두 집단·여러 집단의 차이를 판정하고, 한 변수로 다른 변수를 설명·예측하는 **회귀분석**까지 나아갑니다. 오늘 직접 시뮬레이션으로 만들어 본 p-value 가 그 모든 검정의 공용 언어입니다.